In [111]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys
ROOT = Path.cwd().parent.parent
sys.path.append(str(ROOT))
from src.loading_data.load_data import get_data
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

In [112]:
df = get_data()
df.head()

DataFrame Cleaned Sucessfully...
DataFrame Information:
>>> Columns:
Index(['Gend3', 'Disab3', 'Age9', 'Eth7', 'NSSEC5', 'Educ6', 'MEMS7_ALL',
       'year', 'IMD10', 'nchild', 'VolAny', 'Motiva_POP', 'motivb_POP',
       'motivc_POP', 'motivd_POP', 'WorkStat10', 'active'],
      dtype='str')
>>> Shape (86089, 17)
--------------------------
Columns NOT in Master.csv:
>>> ['nadults', 'motivex2a', 'motivex2b', 'motivex2c', 'motivex2d', 'inclus_a', 'inclus_b', 'inclus_c', 'comm2', 'limfreti1', 'limfreti2', 'limfreti3', 'limfreti4', 'limfreti5', 'limfreti6', 'limfreti7', 'limfreti8']
Unhealthy Columns:
>>> ['motive_POP', 'comm1', 'anxious', 'happy', 'lifesat', 'lone', 'worthw', 'indev', 'indevtry']


,Gend3,Disab3,Age9,Eth7,NSSEC5,Educ6,MEMS7_ALL,year,IMD10,nchild,VolAny,Motiva_POP,motivb_POP,motivc_POP,motivd_POP,WorkStat10,active
19887,1.0,3.0,2.0,4.0,1.0,1.0,0.0,2016/17,2.0,1.0,1.0,1.0,2.0,2.0,4.0,2.0,False
19888,1.0,3.0,4.0,4.0,2.0,1.0,90.0,2016/17,4.0,0.0,0.0,2.0,2.0,4.0,4.0,2.0,False
19889,2.0,3.0,4.0,1.0,1.0,1.0,1200.0,2016/17,7.0,0.0,0.0,2.0,2.0,2.0,4.0,1.0,True
19890,1.0,3.0,4.0,2.0,1.0,1.0,0.0,2016/17,4.0,1.0,0.0,2.0,3.0,4.0,4.0,1.0,False
19891,2.0,1.0,4.0,2.0,1.0,1.0,30.0,2016/17,4.0,1.0,0.0,1.0,1.0,2.0,3.0,2.0,False


In [113]:
dont_encode = ['nchild', 'active', 'MEMS7_ALL', 'year']
discrete_variables = [c for c in df.columns if c not in dont_encode]
# df_encoded = pd.get_dummies(df, columns = discrete_variables, drop_first=True )
for c in discrete_variables:
    print(c, df[c].value_counts().index[0])

reference_categories = [f'{c}_{df[c].value_counts().index[0]}' for c in discrete_variables]
reference_categories

Gend3 2.0
Disab3 3.0
Age9 4.0
Eth7 1.0
NSSEC5 1.0
Educ6 1.0
IMD10 2.0
VolAny 0.0
Motiva_POP 2.0
motivb_POP 2.0
motivc_POP 2.0
motivd_POP 4.0
WorkStat10 1.0


['Gend3_2.0',
 'Disab3_3.0',
 'Age9_4.0',
 'Eth7_1.0',
 'NSSEC5_1.0',
 'Educ6_1.0',
 'IMD10_2.0',
 'VolAny_0.0',
 'Motiva_POP_2.0',
 'motivb_POP_2.0',
 'motivc_POP_2.0',
 'motivd_POP_4.0',
 'WorkStat10_1.0']

In [114]:
encoder = OneHotEncoder(drop='first', handle_unknown = 'ignore', sparse_output = False).set_output(transform='pandas')

In [115]:
encoded = encoder.fit_transform(df[discrete_variables])
df_encoded = pd.concat([df, encoded], axis=1).drop(columns=discrete_variables+['MEMS7_ALL'])
df_encoded.head()

,year,nchild,active,Gend3_2.0,Gend3_3.0,Disab3_2.0,Disab3_3.0,Age9_3.0,Age9_4.0,Age9_5.0,...,motivd_POP_5.0,WorkStat10_2.0,WorkStat10_3.0,WorkStat10_4.0,WorkStat10_5.0,WorkStat10_6.0,WorkStat10_7.0,WorkStat10_8.0,WorkStat10_9.0,WorkStat10_10.0
19887,2016/17,1.0,False,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19888,2016/17,0.0,False,0.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19889,2016/17,0.0,True,1.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19890,2016/17,1.0,False,0.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19891,2016/17,1.0,False,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [116]:
df_encoded_2016 = df_encoded[df_encoded['year'] == '2016/17']
df_encoded_2016 = df_encoded_2016.drop(columns=['year'])

In [117]:
Y = df_encoded_2016['active']
X = df_encoded_2016.drop(columns=['active'])

In [118]:
X.columns

Index(['nchild', 'Gend3_2.0', 'Gend3_3.0', 'Disab3_2.0', 'Disab3_3.0',
       'Age9_3.0', 'Age9_4.0', 'Age9_5.0', 'Age9_6.0', 'Age9_7.0', 'Age9_8.0',
       'Age9_9.0', 'Eth7_2.0', 'Eth7_3.0', 'Eth7_4.0', 'Eth7_5.0', 'Eth7_6.0',
       'Eth7_7.0', 'NSSEC5_2.0', 'NSSEC5_3.0', 'NSSEC5_4.0', 'NSSEC5_5.0',
       'Educ6_2.0', 'Educ6_3.0', 'Educ6_4.0', 'Educ6_5.0', 'Educ6_6.0',
       'IMD10_2.0', 'IMD10_3.0', 'IMD10_4.0', 'IMD10_5.0', 'IMD10_6.0',
       'IMD10_7.0', 'IMD10_8.0', 'IMD10_9.0', 'IMD10_10.0', 'VolAny_1.0',
       'Motiva_POP_2.0', 'Motiva_POP_3.0', 'Motiva_POP_4.0', 'Motiva_POP_5.0',
       'motivb_POP_2.0', 'motivb_POP_3.0', 'motivb_POP_4.0', 'motivb_POP_5.0',
       'motivc_POP_2.0', 'motivc_POP_3.0', 'motivc_POP_4.0', 'motivc_POP_5.0',
       'motivd_POP_2.0', 'motivd_POP_3.0', 'motivd_POP_4.0', 'motivd_POP_5.0',
       'WorkStat10_2.0', 'WorkStat10_3.0', 'WorkStat10_4.0', 'WorkStat10_5.0',
       'WorkStat10_6.0', 'WorkStat10_7.0', 'WorkStat10_8.0', 'WorkStat10_9.0',
    

In [119]:
Xtrain, Xtest, Ytrain, Ytest = train_test_split(X, Y, test_size = 0.25, random_state = 42)
scaler = StandardScaler()
continuous_cols = ['nchild']
Xtrain[continuous_cols] = scaler.fit_transform(Xtrain[continuous_cols])
Xtest[continuous_cols] = scaler.transform(Xtest[continuous_cols])


In [120]:
model = LogisticRegression()
model.fit(Xtrain, Ytrain)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

In [128]:
print(len(model.coef_[0]), len(model.feature_names_in_))


62 62


In [129]:
for idx, name in enumerate(model.feature_names_in_):
    print(f'{name} : {model.coef_[0][idx]}')

nchild : -0.11815260294725356
Gend3_2.0 : -0.07204388304986122
Gend3_3.0 : 0.0
Disab3_2.0 : 0.33830653607745786
Disab3_3.0 : 0.2016554568519471
Age9_3.0 : -0.12997242741180245
Age9_4.0 : -0.055911792585367555
Age9_5.0 : -0.02368665659895195
Age9_6.0 : -0.18900825105740024
Age9_7.0 : -0.4302630293888496
Age9_8.0 : -0.21323853158469985
Age9_9.0 : -0.7539947303639186
Eth7_2.0 : -0.42459729629202564
Eth7_3.0 : -0.9880581575172939
Eth7_4.0 : -0.9031245389323146
Eth7_5.0 : -0.981280354294717
Eth7_6.0 : -0.22401998732411704
Eth7_7.0 : -0.5190672984550289
NSSEC5_2.0 : -0.15241465229007944
NSSEC5_3.0 : -0.4253803497368136
NSSEC5_4.0 : -0.43865628756663994
NSSEC5_5.0 : -0.9672332619486183
Educ6_2.0 : -0.1469049582890792
Educ6_3.0 : -0.2920644536398914
Educ6_4.0 : -0.3002406327820187
Educ6_5.0 : -0.3116204421361882
Educ6_6.0 : -0.48066436100085425
IMD10_2.0 : 0.07680027207714883
IMD10_3.0 : 0.06067847026512626
IMD10_4.0 : -0.08540407877135305
IMD10_5.0 : 0.130230988339855
IMD10_6.0 : 0.1851106847